# Notebook 4 — CMF Unlearning: cmf_static (4a) + Post-Hoc W Calibration (4b)

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Two methods in this notebook, BOTH starting from the SAME cmf_static run:**

**4a. cmf_static (Algorithm 2):** per-epoch `recompute_cmf → freeze W → update encoder only`,  
for 50 epochs total. Default `mean_source='train'` (full D = D_r ∪ D_f, matching paper eq.3/7).  
`mean_source='retain'` is an explicit ablation only.

**4b. cmf_static + post-hoc W calibration:** LOAD the 4a checkpoint (never retrain Stage 1).  
Freeze entire encoder permanently. Promote W to trainable parameter (clean `CMFWeightsTrainable`).  
Run k_posthoc ∈ {2, 5, 10} full epochs of real gradient descent on W only (CE loss on retain_loader).  
Sanity check: Probe/NCC must equal the 4a checkpoint's values (encoder frozen = no change).  

**A.1 fix:** `mean_source='train'` is the paper-faithful default. Never silently use 'retain'.  
**A.2 fix:** cmf_static runs the FULL epoch budget without per-round shortcuts.  
**A.4/A.5 fix:** CMFWeightsTrainable never uses `del _buffers['weight']` hack;  
checkpoints are loadable via `load_state_dict()` into a fresh CMFWeights instance.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/ycgao1/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git','-C',REPO_DIR,'rev-parse','HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-notebook1'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, f'Cannot find cmf_benchmark_config.json'
with open(config_path) as f: NB1_CFG = json.load(f)

DATASET     = NB1_CFG['dataset']
ARCH        = NB1_CFG['arch']
NUM_CLASSES = NB1_CFG['num_classes']
SEEDS       = NB1_CFG['seeds']
RATIOS      = NB1_CFG['ratios']
TEST_MODE   = NB1_CFG.get('test_mode', False)

# CMF experiment matrix
BASE_METHODS   = ['scrub', 'grad_ascent_descent', 'random_label', 'salun']
MEAN_SOURCES   = ['train', 'retain']   # 'train' is paper-faithful; 'retain' is ablation
# A.2 fix: CMF epochs are per-method from Table 4 (NOT all 50).
# Table 4: RL+CMF=4ep, SalUn+CMF=4ep, NegGrad++CMF=3ep, SCRUB+CMF=3ep, UNSIR+CMF=3ep
CMF_EPOCHS_BY_METHOD = {
    'random_label':        2 if TEST_MODE else 4,   # Table 4 line 2390
    'salun':               2 if TEST_MODE else 4,   # Table 4 line 2399
    'grad_ascent_descent': 1 if TEST_MODE else 3,   # Table 4 line 2408
    'scrub':               1 if TEST_MODE else 3,   # Table 4 line 2417
    'tarun':               1 if TEST_MODE else 3,   # Table 4 line 2426
}
K_POSTHOC      = [2, 5, 10]              # post-hoc W calibration epoch counts (NB4b)
PHASE2_DATA    = ['retain_only', 'retain_plus_forget']

CKPT_ROOT = '/kaggle/working/checkpoints/cmf'
os.makedirs(f'{CKPT_ROOT}/cmf_static',  exist_ok=True)
os.makedirs(f'{CKPT_ROOT}/cmf_posthoc', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CMF_EPOCHS_BY_METHOD={CMF_EPOCHS_BY_METHOD}  K_POSTHOC={K_POSTHOC}  device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
full_train = torchvision.datasets.CIFAR10('/kaggle/working/data',train=True,download=True,transform=transform_train)
test_set   = torchvision.datasets.CIFAR10('/kaggle/working/data',train=False,download=True,transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set,batch_size=256,shuffle=False,num_workers=2)
print(f'Train: {len(full_train)}  Test: {len(test_set)}')

In [ ]:
# A.1/A.4/A.5 fix: use ModelModule (CMF model with remove_FC=True)
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import CMFWeightsTrainable

def make_cmf_args(base_method, lr, epochs, mean_source, forget_idx, retain_idx, seed=0):
    # utils.test() ZeroDivisionError (stratified: all classes in unlearn_class) fixed in utils.py.
    forget_classes = list(set(full_train.targets[i] for i in forget_idx))
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=forget_classes,   # real forget classes — used by training perturbation logic
        batch_size=128, test_batch_size=256,
        lr=lr, momentum=0.9, weight_decay=5e-4,
        epochs_or_steps=epochs,
        seed=seed,   # required by random_label, salun (args.seed used for Generator seeds)
        num_retain_samples=len(retain_idx),
        num_forget_samples=len(forget_idx),
        grad_norm_clip=1.0,
        # SVD: Table 4 line 2375: CIFAR-10 alpha_r=1000, alpha_f=30, samples=900
        SVD_alpha_r=1000, SVD_alpha_f=30, SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,   # required by SVD_unlearn -> get_projection_matrix
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        # UNSIR: Table 4 line 2366/2426: impair_lr = same as main lr = 5e-5
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE,
        no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data',
        remove_FC=True, CMFClassifier=True, CMF_momentum=0.9,
        pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0,
        mean_source=mean_source,  # A.1 fix
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

def build_cmf_model(args):
    return ModelModule(args).to(device)

@torch.no_grad()
def cmf_eval_output(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

@torch.no_grad()
def cmf_extract_features(model, loader):
    """Extract CMF-preprocessed features: ẑ = normalize(normalize(f) − μ)."""
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model._preprocess_feats_for_cmf(f)  # A.5 fix
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_probe_on_features(Xtr, ytr, Xte, yte, n_epochs=50):
    d = Xtr.size(1)
    head = nn.Linear(d, NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        pred = head(Xte.to(device)).argmax(1).cpu()
    return pred, yte

def run_ncc_from_features(Xtr, ytr, Xte, yte):
    Xtr_n = F.normalize(Xtr, dim=1)
    means = torch.zeros(NUM_CLASSES, Xtr_n.size(1))
    for c in range(NUM_CLASSES):
        m = (ytr == c)
        if m.any(): means[c] = Xtr_n[m].mean(0)
    means_n = F.normalize(means, dim=1)
    Xte_n   = F.normalize(Xte, dim=1)
    return (Xte_n @ means_n.t()).argmax(1), yte

def acc_split(pred, true, mask):
    ret = (pred[~mask] == true[~mask]).float().mean().item() * 100
    fgt = (pred[mask]  == true[mask]).float().mean().item() * 100
    return ret, fgt

def eval_cmf_three_metrics(model, forget_test_mask, full_train_loader, test_loader):
    model.eval()
    with torch.no_grad():
        preds = torch.cat([model(x.to(device)).argmax(1).cpu() for x,_ in test_loader])
    true = torch.tensor(test_set.targets)
    out_ret, out_fgt = acc_split(preds, true, forget_test_mask)

    Xtr, ytr = cmf_extract_features(model, full_train_loader)
    Xte, yte = cmf_extract_features(model, test_loader)

    lp_pred, lp_true = run_probe_on_features(Xtr, ytr, Xte, yte)
    lp_ret, lp_fgt   = acc_split(lp_pred, lp_true, forget_test_mask)

    ncc_pred, ncc_true = run_ncc_from_features(Xtr, ytr, Xte, yte)
    ncc_ret, ncc_fgt   = acc_split(ncc_pred, ncc_true, forget_test_mask)

    return {'output_retain_acc':out_ret,'output_forget_acc':out_fgt,
            'probe_retain_acc':lp_ret,  'probe_forget_acc':lp_fgt,
            'ncc_retain_acc':ncc_ret,   'ncc_forget_acc':ncc_fgt}

print('CMF helpers ready.')

In [ ]:
# ─── 4a: cmf_static ──────────────────────────────────────────────────────────
# A.2 fix: per-epoch recompute_cmf → freeze W → update encoder only (Algorithm 2)
# Epoch budgets and LRs from Table 4 exclusively (arXiv:2604.08271v1)
from unlearn import unlear_func

# LRs for CMF variants (Table 4, PDF lines 2390–2434)
# Table 4 does not distinguish single-class vs multi-class; one LR per dataset.
HPARAM_SOURCE = 'table4'
CMF_LR = {
    'scrub':               5e-3,   # Table 4 line 2417: CIFAR-10 SCRUB+CMF lr=5×10⁻³
    'grad_ascent_descent': 1e-4,   # Table 4 line 2408: CIFAR-10 NegGrad++CMF lr=1×10⁻⁴
    'random_label':        2e-3,   # Table 4 line 2390: CIFAR-10 RL+CMF lr=2×10⁻³
    'salun':               2e-3,   # Table 4 line 2399: CIFAR-10 SalUn+CMF lr=2×10⁻³
    'tarun':               5e-5,   # Table 4 line 2426: CIFAR-10 UNSIR+CMF lr=5×10⁻⁵
}
# Batch sizes: SCRUB uses 64, others 128 (Table 4)
CMF_BATCH = {
    'scrub': 64,
}
CMF_BATCH_DEFAULT = 128

results_4a = []

for ratio in RATIOS:
    for seed in SEEDS:
        tag_split = f'ratio{ratio}_seed{seed}'
        if TEST_MODE: tag_split += '_testmode'

        fpath = f'{CKPT_ROOT_NB1}/splits/forget_indices_{tag_split}.json'
        rpath = f'{CKPT_ROOT_NB1}/splits/retain_indices_{tag_split}.json'
        with open(fpath) as f: forget_idx = json.load(f)
        with open(rpath) as f: retain_idx = json.load(f)

        theta_o_path = f'{CKPT_ROOT_NB1}/pre_train/theta_o_seed{seed}.pt'
        if TEST_MODE: theta_o_path = theta_o_path.replace('.pt', '_testmode.pt')
        # Load Θ_o state dict (raw weights only)
        ck_o = torch.load(theta_o_path, map_location=device)
        theta_o_state = ck_o.get('model_state_dict', ck_o)

        forget_classes   = list(set(full_train.targets[i] for i in forget_idx))
        # Stratified test mask: same ratio-per-class as NB1 split, applied to test set.
        # Class-membership mask is all-True for stratified splits → breaks retain/forget split.
        import random as _rng_mod; import numpy as _np_mod
        _rng = _rng_mod.Random(seed)
        _test_tgts = _np_mod.array(test_set.targets)
        _forget_test = []
        for _c in range(NUM_CLASSES):
            _cls_test = _np_mod.where(_test_tgts == _c)[0].tolist()
            _n = max(1, int(len(_cls_test) * ratio / 100))
            _forget_test.extend(_rng.sample(_cls_test, _n))
        forget_test_mask = torch.zeros(len(test_set), dtype=torch.bool)
        forget_test_mask[_forget_test] = True

        retain_set = torch.utils.data.Subset(full_train, retain_idx)
        forget_set = torch.utils.data.Subset(full_train, forget_idx)
        retain_loader = torch.utils.data.DataLoader(retain_set, batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(forget_set, batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(full_train, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                tag = (f'{base_method}_cmf_static_{mean_source}_'
                       f'ratio{ratio}_seed{seed}')
                if TEST_MODE: tag += '_testmode'
                ckpt_path = f'{CKPT_ROOT}/cmf_static/{tag}.pt'

                if os.path.exists(ckpt_path):
                    print(f'[4a {tag}] exists — skipping.')
                    ck = torch.load(ckpt_path, map_location=device)
                    results_4a.append(ck['metrics'])
                    continue

                cmf_epochs = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                print(f'\n[4a {tag}] cmf_epochs={cmf_epochs}')
                torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                lr    = CMF_LR.get(base_method, 1e-3)
                batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                args  = make_cmf_args(base_method, lr, cmf_epochs, mean_source,
                                      forget_idx, retain_idx, seed=seed)
                args.batch_size = batch  # override batch size
                if base_method == 'scrub':
                    args.scrub_del_bsz  = 64
                    args.scrub_sgda_bsz = 64
                model = build_cmf_model(args)
                # Load encoder weights from Θ_o (strict=False: CMF head not in Θ_o)
                model.encoder.load_state_dict(theta_o_state, strict=False)

                # A.1 fix: mean_loader = full train if mean_source='train', else retain
                mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                dispatch_key = f'{base_method}_CMF_RemoveFC'
                fn = unlear_func[dispatch_key]

                t0 = time.time()
                try:
                    model = fn(
                        args=args, model=model, device=device,
                        retain_loader=retain_loader, forget_loader=forget_loader,
                        train_loader=mean_loader, test_loader=test_loader,
                        optimizer=None, epochs=cmf_epochs,
                        test_forget_loader=forget_loader,
                        train_dataset=full_train,   # required by tarun_CMF_unlearn, SVD_unlearn
                        val_index=retain_idx,       # required by tarun_CMF_unlearn, SVD_unlearn
                    )
                except Exception as e:
                    print(f'  ERROR: {e}'); continue
                wall_min = (time.time() - t0) / 60

                # Final recompute_cmf from mean_loader
                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                metrics = eval_cmf_three_metrics(model, forget_test_mask, full_train_loader, test_loader)
                metrics.update({'method': base_method, 'mean_source': mean_source,
                                'ratio': ratio, 'seed': seed, 'stage': '4a',
                                'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                                'wall_clock_minutes': wall_min,
                                'hparam_source': HPARAM_SOURCE})

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': {'base_method': base_method, 'mean_source': mean_source,
                               'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                               'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                               'ratio': ratio, 'seed': seed,
                               'hparam_source': HPARAM_SOURCE,
                               'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE},
                    'seed': seed,
                    'metrics': metrics,
                }, ckpt_path)
                print(f'  Saved {ckpt_path}')
                print(f'  out R={metrics["output_retain_acc"]:.2f}% F={metrics["output_forget_acc"]:.2f}%  '
                      f'probe R={metrics["probe_retain_acc"]:.2f}% F={metrics["probe_forget_acc"]:.2f}%')
                results_4a.append(metrics)

df_4a = pd.DataFrame(results_4a)
df_4a.to_csv(f'{CKPT_ROOT}/results_4a_cmf_static.csv', index=False)
print('\n=== 4a cmf_static results saved ===')

In [ ]:
# ─── 4b: post-hoc W calibration ──────────────────────────────────────────────
# LOAD 4a checkpoint (never retrain Stage 1).
# Freeze encoder permanently. Promote W to trainable (CMFWeightsTrainable).
# Run k_posthoc full epochs of gradient descent on W only.
# Sanity: Probe/NCC must match 4a's values (encoder unchanged).

results_4b = []

for ratio in RATIOS:
    for seed in SEEDS:
        tag_split = f'ratio{ratio}_seed{seed}'
        if TEST_MODE: tag_split += '_testmode'
        fpath = f'{CKPT_ROOT_NB1}/splits/forget_indices_{tag_split}.json'
        rpath = f'{CKPT_ROOT_NB1}/splits/retain_indices_{tag_split}.json'
        with open(fpath) as f: forget_idx = json.load(f)
        with open(rpath) as f: retain_idx = json.load(f)

        forget_classes   = list(set(full_train.targets[i] for i in forget_idx))
        # Stratified test mask: same ratio-per-class as NB1 split, applied to test set.
        _rng = _rng_mod.Random(seed)
        _forget_test = []
        for _c in range(NUM_CLASSES):
            _cls_test = _np_mod.where(_test_tgts == _c)[0].tolist()
            _n = max(1, int(len(_cls_test) * ratio / 100))
            _forget_test.extend(_rng.sample(_cls_test, _n))
        forget_test_mask = torch.zeros(len(test_set), dtype=torch.bool)
        forget_test_mask[_forget_test] = True
        retain_set       = torch.utils.data.Subset(full_train, retain_idx)
        forget_set       = torch.utils.data.Subset(full_train, forget_idx)
        retain_loader    = torch.utils.data.DataLoader(retain_set,batch_size=128,shuffle=True,num_workers=2)
        forget_loader    = torch.utils.data.DataLoader(forget_set,batch_size=128,shuffle=True,num_workers=2)
        full_train_loader= torch.utils.data.DataLoader(full_train,batch_size=256,shuffle=False,num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                # Load 4a checkpoint (SOLE source of truth for Stage 1)
                tag_4a = (f'{base_method}_cmf_static_{mean_source}_'
                          f'ratio{ratio}_seed{seed}')
                if TEST_MODE: tag_4a += '_testmode'
                ckpt_4a = f'{CKPT_ROOT}/cmf_static/{tag_4a}.pt'
                if not os.path.exists(ckpt_4a):
                    print(f'[4b] Missing 4a ckpt: {ckpt_4a} — skipping.')
                    continue

                ck_4a = torch.load(ckpt_4a, map_location=device)
                metrics_4a = ck_4a['metrics']

                for k_posthoc in K_POSTHOC:
                    for phase2_data in PHASE2_DATA:
                        tag = (f'{base_method}_cmf_static_posthoc_k{k_posthoc}_'
                               f'{phase2_data}_{mean_source}_ratio{ratio}_seed{seed}')
                        if TEST_MODE: tag += '_testmode'
                        ckpt_path = f'{CKPT_ROOT}/cmf_posthoc/{tag}.pt'

                        if os.path.exists(ckpt_path):
                            print(f'[4b {tag}] exists — skipping.')
                            ck = torch.load(ckpt_path, map_location=device)
                            results_4b.append(ck['metrics'])
                            continue

                        print(f'\n[4b {tag}]')
                        lr   = CMF_LR.get(base_method, 1e-3)
                        batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                        cmf_epochs = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                        args = make_cmf_args(base_method, lr, cmf_epochs, mean_source,
                                             forget_idx, retain_idx)
                        args.batch_size = batch
                        if base_method == 'scrub':
                            args.scrub_del_bsz  = 64
                            args.scrub_sgda_bsz = 64

                        # Load 4a model
                        model = build_cmf_model(args)
                        model.load_state_dict(ck_4a['model_state_dict'], strict=False)
                        model = model.to(device)

                        # Freeze entire encoder
                        for p in model.parameters():
                            p.requires_grad_(False)

                        # Promote W to trainable (clean method — no buffer deletion)
                        tw = CMFWeightsTrainable(model.CMFweights).to(device)
                        tw.promote()

                        # Stage-2 data loader
                        if phase2_data == 'retain_plus_forget':
                            from torch.utils.data import ConcatDataset
                            s2_ds  = ConcatDataset([retain_set, forget_set])
                            s2_ldr = torch.utils.data.DataLoader(s2_ds, batch_size=128, shuffle=True)
                        else:
                            s2_ldr = retain_loader

                        # Stage-2 W optimizer uses same LR as Stage 1 (CMF_LR[base_method])
                        opt_w = optim.SGD([tw.W_param], lr=CMF_LR.get(base_method, 1e-3),
                                          momentum=0.9, weight_decay=1e-4)

                        s2_log = []
                        for ep in range(1, k_posthoc + 1):
                            model.train()  # needed for BN running stats
                            ep_loss = n_batches = 0
                            for xb, yb in s2_ldr:
                                xb, yb = xb.to(device), yb.to(device)
                                opt_w.zero_grad()
                                with torch.no_grad():
                                    f = model.extract_features(xb)
                                    z = model._preprocess_feats_for_cmf(f)
                                logits = tw(z, temperature=1.0)
                                loss   = F.cross_entropy(logits, yb)
                                loss.backward()
                                opt_w.step()
                                ep_loss  += loss.item()
                                n_batches += 1
                                if TEST_MODE: break

                            # Sync W back to buffer so model.forward() and eval use it
                            tw.sync_back()
                            model.eval()

                            out_acc = cmf_eval_output(model, test_loader)
                            # Extract features for Probe/NCC
                            Xtr, ytr = cmf_extract_features(model, full_train_loader)
                            Xte, yte = cmf_extract_features(model, test_loader)
                            lp_pred, _ = run_probe_on_features(Xtr, ytr, Xte, yte)
                            ncc_pred, _ = run_ncc_from_features(Xtr, ytr, Xte, yte)

                            true = torch.tensor(test_set.targets)
                            lp_ret, lp_fgt = acc_split(lp_pred, true, forget_test_mask)
                            ncc_ret, ncc_fgt = acc_split(ncc_pred, true, forget_test_mask)

                            # Sanity check: Probe/NCC must match 4a (encoder frozen)
                            if ep == 1:
                                delta_lp  = abs(lp_ret  - metrics_4a.get('probe_retain_acc',  lp_ret))
                                delta_ncc = abs(ncc_ret - metrics_4a.get('ncc_retain_acc',   ncc_ret))
                                if delta_lp > 1.0 or delta_ncc > 1.0:
                                    print(f'  SANITY FAIL: Probe/NCC differ from 4a!'
                                          f' delta_lp={delta_lp:.2f}% delta_ncc={delta_ncc:.2f}%')
                                else:
                                    print(f'  Sanity OK: Probe/NCC match 4a baseline.')

                            print(f'  [4b ep{ep}] output={out_acc:.2f}%  '
                                  f'probe_ret={lp_ret:.2f}% fgt={lp_fgt:.2f}%  '
                                  f'ncc_ret={ncc_ret:.2f}% fgt={ncc_fgt:.2f}%')
                            s2_log.append({'epoch': ep, 'output_acc': out_acc,
                                           'probe_retain': lp_ret, 'probe_forget': lp_fgt,
                                           'ncc_retain': ncc_ret, 'ncc_forget': ncc_fgt})

                        # Final metrics
                        final_metrics = eval_cmf_three_metrics(model, forget_test_mask,
                                                               full_train_loader, test_loader)
                        final_metrics.update({
                            'method': base_method, 'mean_source': mean_source,
                            'stage': '4b', 'k_posthoc': k_posthoc,
                            'phase2_data': phase2_data,
                            'ratio': ratio, 'seed': seed,
                        })

                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'config': {
                                'base_method': base_method, 'mean_source': mean_source,
                                'k_posthoc': k_posthoc, 'phase2_data': phase2_data,
                                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                'ratio': ratio, 'seed': seed,
                                'source_4a_ckpt': ckpt_4a,
                                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                            },
                            'seed': seed, 'metrics': final_metrics,
                            'stage2_log': s2_log,
                            'metrics_4a_probe_retain': metrics_4a.get('probe_retain_acc'),
                            'metrics_4a_ncc_retain': metrics_4a.get('ncc_retain_acc'),
                        }, ckpt_path)
                        print(f'  Saved {ckpt_path}')
                        results_4b.append(final_metrics)

df_4b = pd.DataFrame(results_4b)
df_4b.to_csv(f'{CKPT_ROOT}/results_4b_cmf_posthoc.csv', index=False)
print('\n=== 4b post-hoc W results saved ===')

In [ ]:
# ─── Pivoted comparison table: 4a vs 4b ──────────────────────────────────────
# Per (base_method, ratio, mean_source): 4a vs 4b for each k_posthoc/phase2_data.
# Shows whether post-hoc W calibration changes Output while Probe/NCC stay frozen.

for base_method in BASE_METHODS:
    for ratio in RATIOS:
        for mean_source in MEAN_SOURCES:
            sub4a = df_4a[(df_4a['method']==base_method) & (df_4a['ratio']==ratio)
                          & (df_4a['mean_source']==mean_source)]
            sub4b = df_4b[(df_4b['method']==base_method) & (df_4b['ratio']==ratio)
                          & (df_4b['mean_source']==mean_source)]
            if sub4a.empty and sub4b.empty: continue

            print(f'\n=== {base_method.upper()} | ratio={ratio} | mean_source={mean_source} | seeds={SEEDS} ===')
            cols = ['output_retain_acc','output_forget_acc','probe_retain_acc',
                    'probe_forget_acc','ncc_retain_acc','ncc_forget_acc']

            def fmt_row(df, label):
                row = [label]
                for c in cols:
                    if c in df.columns:
                        v = df[c].dropna()
                        row.append(f'{v.mean():.1f}±{v.std():.1f}' if len(v) > 0 else 'N/A')
                    else: row.append('N/A')
                return row

            header = ['Variant'] + cols
            rows = [fmt_row(sub4a, 'cmf_static(4a)')]
            for k in K_POSTHOC:
                for ph in PHASE2_DATA:
                    s = sub4b[(sub4b.get('k_posthoc',pd.Series()).eq(k) if 'k_posthoc' in sub4b.columns else False)
                              & (sub4b.get('phase2_data',pd.Series()).eq(ph) if 'phase2_data' in sub4b.columns else False)]
                    if not s.empty:
                        rows.append(fmt_row(s, f'posthoc_k{k}_{ph}'))

            print(pd.DataFrame(rows, columns=header).to_string(index=False))